# MNIST → TensorFlow.js — versión corregida para Google Colab

Este notebook entrena una red neuronal densa para MNIST y convierte el modelo a TensorFlow.js.

**Corrección principal:** la instalación de `tensorflowjs` se hace con `--no-deps` para evitar el conflicto de dependencias de Colab. Después se reinstala el conversor de forma limpia y se parchean sus imports opcionales con una comprobación de sintaxis.

> Ejecuta las celdas **en orden desde el principio**. Si Colab solicita reiniciar el entorno, hazlo y vuelve a ejecutar desde la primera celda.


In [ ]:
# 1. Comprobar entorno
import sys, os, subprocess, site

print("Python:", sys.version)
print("Ejecutable:", sys.executable)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except Exception as e:
    print("No se pudo importar TensorFlow:", repr(e))


In [ ]:
# 2. Reinstalar limpiamente el conversor sin cambiar las dependencias de TensorFlow
# --force-reinstall restaura los archivos originales antes de aplicar los parches.
%pip install -q --no-deps --force-reinstall tensorflowjs==4.22.0

print("TensorFlow.js 4.22.0 reinstalado.")


In [ ]:
# 3. Localizar tensorflowjs sin importarlo
import os, site, glob

candidatos = []

for p in site.getsitepackages():
    candidatos.append(os.path.join(p, "tensorflowjs"))

user_site = site.getusersitepackages()
candidatos.append(os.path.join(user_site, "tensorflowjs"))

tfjs_dirs = [p for p in candidatos if os.path.isdir(p)]

print("Directorios encontrados:")
for p in tfjs_dirs:
    print(" -", p)

if not tfjs_dirs:
    raise FileNotFoundError(
        "No se encontró la carpeta tensorflowjs. "
        "La instalación de la celda anterior no terminó correctamente."
    )

TFJS_DIR = tfjs_dirs[0]
print("\nTFJS_DIR =", TFJS_DIR)


In [ ]:
# 4. Parche de compatibilidad para NumPy moderno
# Usamos una expresión regular para reemplazar np.object, pero no np.object_.

import os, re

archivos_np = [
    os.path.join(TFJS_DIR, "converters", "read_weights.py"),
    os.path.join(TFJS_DIR, "converters", "write_weights.py"),
]

for archivo in archivos_np:
    if os.path.isfile(archivo):
        with open(archivo, "r", encoding="utf-8") as f:
            txt = f.read()

        nuevo = re.sub(r"\bnp\.object\b", "object", txt)

        if nuevo != txt:
            with open(archivo, "w", encoding="utf-8") as f:
                f.write(nuevo)
            print("Parche NumPy aplicado:", archivo)
        else:
            print("No fue necesario modificar:", archivo)


In [ ]:
# 5. Parchear imports opcionales de forma segura
# IMPORTANTE: sustituimos cada import por una asignación simple.
# Así evitamos insertar secuencias literales "\n" que causan SyntaxError.

import os

archivo = os.path.join(TFJS_DIR, "converters", "tf_saved_model_conversion_v2.py")

if not os.path.isfile(archivo):
    raise FileNotFoundError(f"No se encontró el archivo del conversor: {archivo}")

with open(archivo, "r", encoding="utf-8") as f:
    lineas = f.readlines()

salida = []
parches = []

for linea in lineas:
    stripped = linea.strip()
    indent = linea[:len(linea) - len(linea.lstrip())]

    if stripped in (
        "import tensorflow_decision_forests",
        "import tensorflow_decision_forests as tfdf",
    ):
        salida.append(indent + "tensorflow_decision_forests = None\n")
        parches.append("tensorflow_decision_forests")
    elif stripped in (
        "import tensorflow_hub as hub",
        "import tensorflow_hub",
    ):
        salida.append(indent + "hub = None\n")
        parches.append("tensorflow_hub")
    else:
        salida.append(linea)

texto_nuevo = "".join(salida)

# Comprobación de sintaxis antes de guardar el parche
compile(texto_nuevo, archivo, "exec")

with open(archivo, "w", encoding="utf-8") as f:
    f.write(texto_nuevo)

print("Imports opcionales revisados:", parches if parches else "No se encontraron imports exactos.")
print("El archivo parcheado pasó la comprobación de sintaxis.")


**Nota de compatibilidad:** la versión actual de Python/TensorFlow de Colab puede presentar otros errores internos del conversor. Este parche evita el error por la dependencia opcional `tensorflow_decision_forests`; no puede garantizar compatibilidad completa del conversor en Python 3.13. Si aparece otro `ModuleNotFoundError` dentro de `tensorflowjs`, no ejecutes las celdas finales: habrá que convertir en un entorno compatible de Python 3.10/3.11.\n

## 6. Cargar MNIST

Se utiliza `tf.keras.datasets.mnist`, evitando `tensorflow_datasets`.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Añadir canal: 28 x 28 x 1
x_train = x_train[..., np.newaxis]
x_test  = x_test[..., np.newaxis]

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)


In [ ]:
# 7. Visualizar algunos números
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(f"Etiqueta: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()


# 8. Crear la red neuronal densa

Arquitectura:

**28×28×1 → Flatten → Dense(50) → Dense(50) → Dense(10)**

La salida tiene 10 neuronas, una por cada dígito del 0 al 9.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(50, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 9. Entrenar
history = model.fit(
    x_train,
    y_train,
    epochs=60,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)


In [ ]:
# 10. Graficar entrenamiento
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Entrenamiento")
plt.plot(history.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Entrenamiento")
plt.plot(history.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# 11. Evaluar
loss, acc = model.evaluate(x_test, y_test, verbose=0)

print(f"Loss     : {loss:.6f}")
print(f"Accuracy : {acc:.6f}")
print(f"Accuracy : {acc*100:.2f}%")


In [ ]:
# 12. Guardar modelo Keras
MODEL_H5 = "/content/numeros_regular.h5"

model.save(MODEL_H5)

print("Modelo guardado en:", MODEL_H5)
print("Existe:", os.path.exists(MODEL_H5))


# 13. Convertir a TensorFlow.js

Esta celda **no usa `import tensorflowjs`** directamente. Ejecuta el conversor como programa.

Primero comprueba que exista `tensorflowjs_converter`.


In [ ]:
# 13.1 Verificar el ejecutable del conversor
import shutil

converter = shutil.which("tensorflowjs_converter")

print("tensorflowjs_converter =", converter)

if converter is None:
    # Buscarlo manualmente en los directorios de Python
    posibles = []
    for p in site.getsitepackages():
        posibles.extend(glob.glob(os.path.join(p, "tensorflowjs", "**", "tensorflowjs_converter"), recursive=True))
        posibles.extend(glob.glob(os.path.join(p, "**", "tensorflowjs_converter"), recursive=True))

    posibles = [p for p in posibles if os.path.isfile(p)]

    print("Búsqueda manual:")
    for p in posibles[:20]:
        print(" -", p)

    if posibles:
        converter = posibles[0]

if converter is None:
    raise FileNotFoundError(
        "No se encontró tensorflowjs_converter. "
        "Revisa la celda 2 y la celda 3."
    )


In [ ]:
# 14. Convertir H5 → TensorFlow.js Layers Model

OUT_DIR = "/content/modelo_tfjs"

if os.path.exists(OUT_DIR):
    import shutil
    shutil.rmtree(OUT_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

cmd = [
    converter,
    "--input_format=keras",
    MODEL_H5,
    OUT_DIR
]

print("Ejecutando:")
print(" ".join(cmd))

resultado = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    raise RuntimeError(
        "La conversión falló. Código de salida: "
        + str(resultado.returncode)
    )

print("Conversión terminada correctamente.")


In [ ]:
# 15. Verificar que model.json y los pesos existan
import os, glob

model_json = os.path.join(OUT_DIR, "model.json")
bin_files = glob.glob(os.path.join(OUT_DIR, "*.bin"))

print("model.json:", model_json)
print("Existe model.json:", os.path.isfile(model_json))
print("Archivos .bin:", len(bin_files))

for f in bin_files:
    print(" -", os.path.basename(f), os.path.getsize(f), "bytes")

if not os.path.isfile(model_json):
    raise FileNotFoundError("No se generó model.json.")

if len(bin_files) == 0:
    raise FileNotFoundError("No se generaron archivos .bin de pesos.")

print("\n✓ Modelo TensorFlow.js validado.")


In [ ]:
# 16. Crear ZIP y descargar
import zipfile

ZIP_PATH = "/content/modelo_mnist_tensorflowjs.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT_DIR):
        for file in files:
            path = os.path.join(root, file)
            arcname = os.path.relpath(path, OUT_DIR)
            z.write(path, arcname)

print("ZIP creado:", ZIP_PATH)
print("Tamaño:", os.path.getsize(ZIP_PATH), "bytes")

from google.colab import files
files.download(ZIP_PATH)


# 17. Resultado esperado

Al final se descargará:

`modelo_mnist_tensorflowjs.zip`

Dentro del ZIP deben aparecer:

- `model.json`
- uno o varios archivos `.bin`

Esos archivos son los que puedes cargar posteriormente con TensorFlow.js.
